In [1]:
# TEMPORAL PART1 — CELL 01
# RAW TIMESTAMP SOURCE DISCOVERY & READINESS
#
# READ ONLY | NO MODEL TRAINING | NO DATA MODIFICATION
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MODEL_PATH = (
    ROOT / "04_Modeling" / "outputs"
    / "leakage_grouping"
    / "modeling_labeled_348_grouped.csv"
)

RAW_DIR = ROOT / "01_Raw Data" / "outputs" / "raw_csv"

EXPECTED_MODEL_SHA = (
    "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

assert MODEL_PATH.is_file(), "Dataset Modeling tidak ditemukan"
assert RAW_DIR.is_dir(), "Folder raw_csv tidak ditemukan"

assert sha256_file(MODEL_PATH) == EXPECTED_MODEL_SHA, (
    "Dataset Modeling berbeda dari versi terkunci"
)

model = pd.read_csv(
    MODEL_PATH,
    encoding="utf-8-sig",
    dtype={"id": "string"},
    keep_default_na=False
)

assert len(model) == 348
assert model["id"].is_unique

assert {"id", "case", "sentiment_label"}.issubset(model.columns)

raw_files = sorted(RAW_DIR.rglob("*.csv"))

assert raw_files, "Tidak ada raw CSV ditemukan"

target_ids = set(model["id"].astype(str))

matches = []
files_with_createdat = 0
files_without_createdat = []
read_errors = []
raw_rows_scanned = 0

for path in raw_files:
    try:
        header = pd.read_csv(
            path,
            nrows=0,
            encoding="utf-8-sig"
        )

        if not {"id", "createdAt"}.issubset(header.columns):
            files_without_createdat.append(
                str(path.relative_to(ROOT))
            )
            continue

        files_with_createdat += 1

        raw = pd.read_csv(
            path,
            usecols=["id", "createdAt"],
            encoding="utf-8-sig",
            dtype={"id": "string", "createdAt": "string"},
            keep_default_na=False,
            on_bad_lines="error"
        )

        raw_rows_scanned += len(raw)

        subset = raw.loc[
            raw["id"].isin(target_ids)
        ].copy()

        if not subset.empty:
            subset["source_file"] = str(
                path.relative_to(ROOT)
            ).replace("\\", "/")

            matches.append(subset)

    except Exception as error:
        read_errors.append({
            "file": str(path.relative_to(ROOT)),
            "error": str(error)[:300]
        })

if matches:
    matched = pd.concat(matches, ignore_index=True)
else:
    matched = pd.DataFrame(
        columns=["id", "createdAt", "source_file"]
    )

matched["createdAt"] = (
    matched["createdAt"]
    .astype("string")
    .str.strip()
)

matched["parsed_utc"] = pd.to_datetime(
    matched["createdAt"],
    utc=True,
    errors="coerce"
)

matched["timestamp_valid"] = matched["parsed_utc"].notna()

# Count different VALID timestamps for each tweet ID.
valid_matches = matched.loc[matched["timestamp_valid"]]

timestamp_conflicts = (
    valid_matches.groupby("id")["parsed_utc"]
    .nunique()
)

conflict_ids = set(
    timestamp_conflicts[timestamp_conflicts > 1].index
)

valid_ids = set(valid_matches["id"].astype(str))
matched_ids = set(matched["id"].astype(str))

missing_ids = target_ids - matched_ids
matched_without_valid_time = matched_ids - valid_ids

print("=" * 72)
print("05_TEMPORAL — PART1 TIMESTAMP READINESS")
print("=" * 72)

print("Locked Modeling rows        :", len(model))
print("Unique Modeling IDs         :", model["id"].nunique())
print("Raw CSV files found         :", len(raw_files))
print("Files with createdAt        :", files_with_createdat)
print("Raw rows scanned            :", raw_rows_scanned)
print("Raw matching records        :", len(matched))
print("Unique matched IDs          :", len(matched_ids))
print("Unique valid timestamp IDs  :", len(valid_ids))
print("Missing raw IDs             :", len(missing_ids))
print("Matched IDs no valid time   :", len(matched_without_valid_time))
print("Conflicting timestamp IDs   :", len(conflict_ids))
print("Raw file read errors        :", len(read_errors))

print("\nMODEL DISTRIBUTION BY CASE")
display(
    model.groupby("case")
    .agg(
        tweet_count=("id", "nunique")
    )
    .reset_index()
)

print("\nTIMESTAMP COVERAGE BY CASE")
coverage = model[["id", "case"]].copy()

coverage["found_raw"] = coverage["id"].isin(matched_ids)
coverage["valid_timestamp"] = coverage["id"].isin(valid_ids)
coverage["timestamp_conflict"] = coverage["id"].isin(conflict_ids)

display(
    coverage.groupby("case")
    .agg(
        tweets=("id", "count"),
        found_raw=("found_raw", "sum"),
        valid_timestamp=("valid_timestamp", "sum"),
        conflicting_timestamp=("timestamp_conflict", "sum")
    )
    .reset_index()
)

if not valid_matches.empty:
    print("\nEARLIEST / LATEST RAW TIMESTAMPS (UTC)")
    print("Earliest:", valid_matches["parsed_utc"].min())
    print("Latest  :", valid_matches["parsed_utc"].max())

    first_valid = (
        valid_matches.sort_values(["id", "parsed_utc"])
        .drop_duplicates("id")
        [["id", "parsed_utc"]]
    )

    date_preview = model[["id", "case"]].merge(
        first_valid,
        on="id",
        how="left",
        validate="one_to_one"
    )

    date_preview["date_wib"] = (
        date_preview["parsed_utc"]
        .dt.tz_convert("Asia/Jakarta")
        .dt.date
    )

    print("\nDATE COVERAGE BY CASE (WIB)")
    display(
        date_preview.groupby("case")
        .agg(
            earliest=("date_wib", "min"),
            latest=("date_wib", "max"),
            rows_with_dates=("parsed_utc", "count")
        )
        .reset_index()
    )

if read_errors:
    print("\nRAW FILE READ ERRORS (FIRST 10)")
    display(pd.DataFrame(read_errors).head(10))

if files_without_createdat:
    print("\nFILES WITHOUT TIMESTAMP COLUMN (FIRST 10)")
    print("\n".join(files_without_createdat[:10]))

print("\nFINAL PREFLIGHT DECISION")

if (
    len(valid_ids) == 348
    and len(conflict_ids) == 0
    and len(read_errors) == 0
):
    print("TEMPORAL TIMESTAMP PREFLIGHT: PASS")
else:
    print("TEMPORAL TIMESTAMP PREFLIGHT: REVIEW REQUIRED")

print("READ-ONLY SOURCE SCAN COMPLETE")
print("=" * 72)


05_TEMPORAL — PART1 TIMESTAMP READINESS
Locked Modeling rows        : 348
Unique Modeling IDs         : 348
Raw CSV files found         : 65
Files with createdAt        : 57
Raw rows scanned            : 1388
Raw matching records        : 831
Unique matched IDs          : 348
Unique valid timestamp IDs  : 348
Missing raw IDs             : 0
Matched IDs no valid time   : 0
Conflicting timestamp IDs   : 0
Raw file read errors        : 0

MODEL DISTRIBUTION BY CASE


C:\Users\Yusak SR\AppData\Local\Temp\ipykernel_1372\4237529174.py:124: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  matched["parsed_utc"] = pd.to_datetime(


,case,tweet_count
0,lombok_2018,101
1,palu_2018,105
2,sumatra_2025,142



TIMESTAMP COVERAGE BY CASE


,case,tweets,found_raw,valid_timestamp,conflicting_timestamp
0,lombok_2018,101,101,101,0
1,palu_2018,105,105,105,0
2,sumatra_2025,142,142,142,0



EARLIEST / LATEST RAW TIMESTAMPS (UTC)
Earliest: 2018-08-06 07:40:02+00:00
Latest  : 2025-12-04 14:43:34+00:00

DATE COVERAGE BY CASE (WIB)


,case,earliest,latest,rows_with_dates
0,lombok_2018,2018-08-06,2018-08-14,101
1,palu_2018,2018-09-29,2018-10-03,105
2,sumatra_2025,2025-11-29,2025-12-04,142



FILES WITHOUT TIMESTAMP COLUMN (FIRST 10)
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_539338a38118f5470d1b.csv
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_7085e181db6c8ff0911a.csv
01_Raw Data\outputs\raw_csv\lombok_2018\lombok_2018_971bf25151d9f4419d91.csv
01_Raw Data\outputs\raw_csv\palu_2018\palu_2018_0ac1c9ee684c47c748ff.csv
01_Raw Data\outputs\raw_csv\palu_2018\palu_2018_b4187151cd0d81be2864.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_04ed36216ec543478f4a.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_4f9a9ad2f4ca2fcd88d6.csv
01_Raw Data\outputs\raw_csv\sumatra_2025\sumatra_2025_a2a75c0483636c3341eb.csv

FINAL PREFLIGHT DECISION
TEMPORAL TIMESTAMP PREFLIGHT: PASS
READ-ONLY SOURCE SCAN COMPLETE


In [2]:
# TEMPORAL PART2 — CELL 02
# TIMESTAMP CANONICALIZATION & IMMUTABLE DATASET LOCK
#
# READ ONLY SOURCES | ZERO MODEL TRAINING
# STRICT TIMESTAMP CONFLICT CHECK
# SHA256 PROVENANCE | EXCLUSIVE OUTPUT CREATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MODEL_PATH = (
    ROOT / "04_Modeling" / "outputs"
    / "leakage_grouping"
    / "modeling_labeled_348_grouped.csv"
)

RAW_DIR = ROOT / "01_Raw Data" / "outputs" / "raw_csv"

OUT_DIR = (
    ROOT / "05_Temporal" / "outputs"
    / "timestamp_canonicalization"
)

CANONICAL_PATH = OUT_DIR / "temporal_labeled_348_locked.csv"
PROVENANCE_PATH = OUT_DIR / "timestamp_source_provenance.csv"
COVERAGE_PATH = OUT_DIR / "timestamp_coverage_by_case.csv"
MANIFEST_PATH = OUT_DIR / "temporal_timestamp_lock_manifest.json"
AUDIT_PATH = OUT_DIR / "part2_timestamp_lock_audit.json"

OUTPUTS = [
    CANONICAL_PATH,
    PROVENANCE_PATH,
    COVERAGE_PATH,
    MANIFEST_PATH,
    AUDIT_PATH,
]

EXPECTED_MODEL_SHA = (
    "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7"
)

EXPECTED_CASES = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142,
}

EXPECTED_LABELS = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15,
}

TWITTER_DATE_FORMAT = "%a %b %d %H:%M:%S %z %Y"

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relpath(path):
    return path.relative_to(ROOT).as_posix()

def write_json_temp(path, payload):
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

print("=" * 76)
print("TEMPORAL PART2 — TIMESTAMP CANONICALIZATION")
print("=" * 76)

# ------------------------------------------------------------
# A. PREFLIGHT — LOCKED INPUT AND OUTPUT COLLISION
# ------------------------------------------------------------

assert MODEL_PATH.is_file(), "Locked Modeling file missing"
assert RAW_DIR.is_dir(), "Raw CSV directory missing"

for output in OUTPUTS:
    if output.exists():
        raise FileExistsError(
            f"Output already exists; refusing overwrite: {output}"
        )

assert sha256_file(MODEL_PATH) == EXPECTED_MODEL_SHA, (
    "Locked Modeling dataset SHA256 mismatch"
)

model = pd.read_csv(
    MODEL_PATH,
    encoding="utf-8-sig",
    dtype={"id": "string"},
    keep_default_na=False,
)

assert len(model) == 348
assert model["id"].is_unique

required_model = {
    "id", "case", "sentiment_label",
    "leakage_group_id"
}
assert required_model.issubset(model.columns)

assert model["case"].value_counts().to_dict() == EXPECTED_CASES
assert model["sentiment_label"].value_counts().to_dict() == EXPECTED_LABELS

raw_files = sorted(RAW_DIR.rglob("*.csv"))
assert len(raw_files) > 0

# Hash every file before reading. All raw sources are immutable.
source_hashes = {
    relpath(MODEL_PATH): sha256_file(MODEL_PATH)
}
source_hashes.update({
    relpath(path): sha256_file(path)
    for path in raw_files
})

target_keys = set(
    zip(
        model["case"].astype(str),
        model["id"].astype(str)
    )
)

print("Locked rows:", len(model))
print("Raw CSV files:", len(raw_files))

# ------------------------------------------------------------
# B. SCAN RAW FILES AND STRICTLY PARSE TIMESTAMPS
# ------------------------------------------------------------

raw_matches = []
files_without_timestamp = []
read_errors = []
raw_rows_scanned = 0
files_with_timestamp = 0
matched_files = set()

for path in raw_files:
    source_name = relpath(path)

    try:
        header = pd.read_csv(
            path, nrows=0, encoding="utf-8-sig"
        )

        if not {"id", "createdAt"}.issubset(header.columns):
            files_without_timestamp.append(source_name)
            continue

        files_with_timestamp += 1

        raw = pd.read_csv(
            path,
            usecols=["id", "createdAt"],
            dtype={"id": "string", "createdAt": "string"},
            encoding="utf-8-sig",
            keep_default_na=False,
            on_bad_lines="error",
        )

        raw_rows_scanned += len(raw)

        # Expected path: raw_csv/<case>/<filename>.csv
        source_case = path.parent.name

        if source_case not in EXPECTED_CASES:
            raise AssertionError(
                f"Unknown source case: {source_case}"
            )

        raw["id"] = raw["id"].str.strip()
        raw["createdAt"] = raw["createdAt"].str.strip()

        subset = raw.loc[
            raw["id"].isin(model["id"])
        ].copy()

        # Enforce that each matched tweet belongs to its
        # original event/case; do not accept cross-case IDs.
        if not subset.empty:
            subset["case"] = source_case

            pairs = list(zip(
                subset["case"].astype(str),
                subset["id"].astype(str)
            ))

            if not all(key in target_keys for key in pairs):
                raise AssertionError(
                    f"Cross-case ID mismatch in {source_name}"
                )

            subset["source_file"] = source_name
            subset["source_file_sha256"] = source_hashes[source_name]
            subset["source_row_number"] = (
                subset.index.to_numpy() + 2
            )

            raw_matches.append(subset)
            matched_files.add(source_name)

    except Exception as exc:
        read_errors.append({
            "file": source_name,
            "error": str(exc)[:500]
        })

assert not read_errors, (
    f"Raw source errors detected: {read_errors[:5]}"
)

assert raw_matches, "No matching timestamp sources found"

matches = pd.concat(raw_matches, ignore_index=True)

assert matches["id"].notna().all()
assert (matches["createdAt"] != "").all(), (
    "Matching raw records contain blank timestamps"
)

# A fixed parser prevents pandas/dateutil guesswork.
# Twitter API-style: Mon Aug 13 04:49:43 +0000 2018
matches["created_at_utc"] = pd.to_datetime(
    matches["createdAt"],
    format=TWITTER_DATE_FORMAT,
    errors="coerce",
    utc=True
)

invalid_time = matches["created_at_utc"].isna()

assert not invalid_time.any(), (
    "Unparseable raw timestamps: "
    + repr(
        matches.loc[
            invalid_time,
            ["id", "createdAt", "source_file"]
        ].head(10).to_dict("records")
    )
)

# Full precision: timestamps may differ by seconds.
timestamps_per_id = (
    matches.groupby("id")["created_at_utc"]
    .nunique()
)

conflict_ids = (
    timestamps_per_id[timestamps_per_id > 1]
    .index.tolist()
)

assert not conflict_ids, (
    f"Contradictory timestamps for IDs: {conflict_ids[:10]}"
)

matched_ids = set(matches["id"].astype(str))
model_ids = set(model["id"].astype(str))

assert matched_ids == model_ids, (
    f"Missing IDs: {sorted(model_ids - matched_ids)[:10]}"
)

assert matches.groupby("id")["case"].nunique().max() == 1

print("Matching raw occurrences:", len(matches))
print("Unique matched IDs:", len(matched_ids))
print("Timestamp conflicts:", len(conflict_ids))
print("Raw read errors:", len(read_errors))

# ------------------------------------------------------------
# C. DETERMINISTIC CANONICAL TIMESTAMP SELECTION
# ------------------------------------------------------------

# Choose source deterministically only AFTER proving every
# occurrence of the same ID has exactly the same timestamp.
matches = matches.sort_values(
    ["case", "id", "source_file", "source_row_number"],
    kind="mergesort"
).reset_index(drop=True)

matches["is_canonical_source"] = (
    ~matches.duplicated(subset=["id"])
)

primary = matches.loc[
    matches["is_canonical_source"]
].copy()

assert len(primary) == 348
assert primary["id"].is_unique

source_occurrences = (
    matches.groupby("id").size().rename(
        "raw_occurrence_count"
    )
)

source_file_counts = (
    matches.groupby("id")["source_file"].nunique()
    .rename("raw_source_file_count")
)

primary = primary.set_index("id")

primary = primary.join(
    source_occurrences
).join(
    source_file_counts
)

# Preserve the 348 locked Modeling rows and their order.
canonical = model[[
    "case",
    "id",
    "sentiment_label",
    "leakage_group_id"
]].copy()

canonical = canonical.merge(
    primary.reset_index()[[
        "id", "createdAt", "created_at_utc",
        "source_file", "source_file_sha256",
        "source_row_number", "raw_occurrence_count",
        "raw_source_file_count"
    ]],
    on="id",
    how="left",
    validate="one_to_one",
    sort=False
)

assert len(canonical) == 348
assert canonical["id"].is_unique
assert canonical["created_at_utc"].notna().all()

# UTC ISO 8601, +00:00 explicitly.
canonical["created_at_utc"] = (
    canonical["created_at_utc"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)

# Convert separately from canonical UTC strings.
utc_dt = pd.to_datetime(
    canonical["created_at_utc"],
    utc=True,
    format="%Y-%m-%dT%H:%M:%S%z"
)

wib_dt = utc_dt.dt.tz_convert("Asia/Jakarta")

canonical["created_at_wib"] = (
    wib_dt.dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)
canonical["date_wib"] = wib_dt.dt.strftime("%Y-%m-%d")
canonical["hour_wib"] = wib_dt.dt.hour.astype(int)

canonical = canonical.rename(columns={
    "createdAt": "raw_createdAt",
    "source_file": "canonical_source_file",
    "source_file_sha256": "canonical_source_sha256",
    "source_row_number": "canonical_source_row_number"
})

canonical = canonical[[
    "id", "case", "sentiment_label", "leakage_group_id",
    "raw_createdAt", "created_at_utc", "created_at_wib",
    "date_wib", "hour_wib",
    "canonical_source_file", "canonical_source_sha256",
    "canonical_source_row_number",
    "raw_occurrence_count", "raw_source_file_count"
]]

# Verify labels, groups and case have not been altered.
for col in [
    "id", "case", "sentiment_label", "leakage_group_id"
]:
    assert np.array_equal(
        canonical[col].astype(str).to_numpy(),
        model[col].astype(str).to_numpy()
    ), f"Locked value mismatch: {col}"

# Strictly verify the timezone offset relationship.
assert (
    wib_dt.dt.tz_convert("UTC") == utc_dt
).all()

# ------------------------------------------------------------
# D. SOURCE PROVENANCE FOR EVERY RAW OCCURRENCE
# ------------------------------------------------------------

provenance = matches[[
    "id", "case", "createdAt", "created_at_utc",
    "source_file", "source_file_sha256",
    "source_row_number", "is_canonical_source"
]].copy()

provenance["created_at_utc"] = (
    provenance["created_at_utc"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
)

provenance = provenance.rename(columns={
    "createdAt": "raw_createdAt"
})

assert provenance["is_canonical_source"].sum() == 348
assert set(provenance["id"]) == model_ids

# ------------------------------------------------------------
# E. TEMPORAL COVERAGE BY CASE
# ------------------------------------------------------------

coverage = (
    canonical.groupby("case", as_index=False)
    .agg(
        tweet_count=("id", "nunique"),
        earliest_date_wib=("date_wib", "min"),
        latest_date_wib=("date_wib", "max"),
        raw_occurrences=("raw_occurrence_count", "sum")
    )
    .sort_values("case")
    .reset_index(drop=True)
)

assert dict(zip(
    coverage["case"], coverage["tweet_count"]
)) == EXPECTED_CASES

print("\nTEMPORAL COVERAGE")
print(coverage.to_string(index=False))

# ------------------------------------------------------------
# F. MANIFEST AND AUDIT
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

manifest = {
    "stage": "05_Temporal",
    "part": "2 Timestamp Canonicalization & Lock",
    "generated_at_utc": now,
    "source_commit":
        "4e29215d8a7ee1b5d98943b2edcb70f7538060aa",
    "dataset_id": "TEMPORAL_LABELED_348_V1",
    "modeling_source_sha256": EXPECTED_MODEL_SHA,
    "timestamp_source": "raw_csv.createdAt",
    "raw_timestamp_format": TWITTER_DATE_FORMAT,
    "canonical_timezone": "UTC",
    "analysis_timezone": "Asia/Jakarta",
    "canonicalization_policy": (
        "Accept only if all raw occurrences of each ID "
        "have an identical valid timestamp; choose "
        "lexicographically first source file and lowest "
        "source row as deterministic provenance."
    ),
    "modeling_rows": 348,
    "canonical_rows": len(canonical),
    "unique_ids": int(canonical["id"].nunique()),
    "matching_raw_occurrences": len(matches),
    "raw_files_scanned": len(raw_files),
    "raw_files_with_timestamp": files_with_timestamp,
    "raw_files_without_timestamp":
        files_without_timestamp,
    "matched_raw_files": sorted(matched_files),
    "raw_rows_scanned": raw_rows_scanned,
    "case_distribution": EXPECTED_CASES,
    "label_distribution": EXPECTED_LABELS,
    "timestamp_conflicts": len(conflict_ids),
    "missing_timestamp_ids": 0,
    "source_sha256": source_hashes,
    "output_sha256": {},
    "status": "TEMPORAL_TIMESTAMP_LOCK_PREPARED"
}

audit = {
    "stage": "05_Temporal",
    "part": "2 Timestamp Integrity Audit",
    "generated_at_utc": now,
    "modeling_rows_preserved": True,
    "modeling_ids_preserved": True,
    "modeling_cases_preserved": True,
    "modeling_labels_preserved": True,
    "leakage_groups_preserved": True,
    "raw_timestamp_parse_errors": 0,
    "raw_file_read_errors": len(read_errors),
    "missing_timestamp_ids": 0,
    "conflicting_timestamp_ids": 0,
    "canonical_rows": 348,
    "canonical_unique_ids": 348,
    "source_dataset_modified": False,
    "raw_files_modified": False,
    "model_training_performed": False,
    "status": "TEMPORAL_PART2_TIMESTAMP_LOCK_PASS"
}

# ------------------------------------------------------------
# G. IMMUTABLE EXPORT — NO OVERWRITE
# ------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

for output in OUTPUTS:
    if output.exists():
        raise FileExistsError(
            f"Refusing overwrite: {output}"
        )

token = uuid.uuid4().hex[:10]
tmp = {
    path: OUT_DIR / f".part2_{token}_{i}.tmp"
    for i, path in enumerate(OUTPUTS)
}
created = []

try:
    canonical.to_csv(
        tmp[CANONICAL_PATH],
        index=False, encoding="utf-8-sig"
    )
    provenance.to_csv(
        tmp[PROVENANCE_PATH],
        index=False, encoding="utf-8-sig"
    )
    coverage.to_csv(
        tmp[COVERAGE_PATH],
        index=False, encoding="utf-8-sig"
    )

    # Verify the saved CSV, not just the in-memory frame.
    check = pd.read_csv(
        tmp[CANONICAL_PATH],
        encoding="utf-8-sig",
        dtype={"id": "string"},
        keep_default_na=False
    )

    assert len(check) == 348
    assert check["id"].is_unique
    assert set(check["id"]) == model_ids

    manifest["output_sha256"] = {
        path.name: sha256_file(tmp[path])
        for path in [
            CANONICAL_PATH, PROVENANCE_PATH, COVERAGE_PATH
        ]
    }
    manifest["status"] = "TEMPORAL_TIMESTAMP_LOCK_FINAL"

    write_json_temp(tmp[MANIFEST_PATH], manifest)

    audit["output_sha256"] = {
        **manifest["output_sha256"],
        MANIFEST_PATH.name: sha256_file(tmp[MANIFEST_PATH])
    }

    write_json_temp(tmp[AUDIT_PATH], audit)

    # All source hashes must still match before writing.
    for src in [MODEL_PATH] + raw_files:
        assert sha256_file(src) == source_hashes[relpath(src)], (
            f"Source modified during execution: {src}"
        )

    for output in OUTPUTS:
        with output.open("xb") as handle:
            handle.write(tmp[output].read_bytes())
        created.append(output)

except Exception:
    for output in created:
        if output.exists():
            output.unlink()
    raise

finally:
    for temporary in tmp.values():
        if temporary.exists():
            temporary.unlink()

# ------------------------------------------------------------
# H. FINAL OUTPUT / SOURCE VERIFICATION
# ------------------------------------------------------------

for output_name, expected_sha in audit["output_sha256"].items():
    actual = sha256_file(OUT_DIR / output_name)
    assert actual == expected_sha, (
        f"Output SHA mismatch: {output_name}"
    )

for src in [MODEL_PATH] + raw_files:
    assert sha256_file(src) == source_hashes[relpath(src)]

assert json.loads(
    AUDIT_PATH.read_text(encoding="utf-8")
)["status"] == "TEMPORAL_PART2_TIMESTAMP_LOCK_PASS"

print("\n" + "=" * 76)
print("TEMPORAL PART2 — FINAL AUDIT")
print("=" * 76)
print("Canonical tweet rows       :", len(canonical))
print("Unique tweet IDs           :", canonical["id"].nunique())
print("Matching raw occurrences   :", len(provenance))
print("Canonical timestamp missing:", 0)
print("Timestamp conflicts        :", 0)
print("Case distribution preserved:", True)
print("Label distribution preserved:", True)
print("Raw sources modified       :", False)
print("Training performed         :", False)

print("\nARTIFACTS:")
for output in OUTPUTS:
    print("-", output)

print("\nFINAL STATUS:", audit["status"])
print("=" * 76)


TEMPORAL PART2 — TIMESTAMP CANONICALIZATION
Locked rows: 348
Raw CSV files: 65
Matching raw occurrences: 831
Unique matched IDs: 348
Timestamp conflicts: 0
Raw read errors: 0

TEMPORAL COVERAGE
        case  tweet_count earliest_date_wib latest_date_wib  raw_occurrences
 lombok_2018          101        2018-08-06      2018-08-14              221
   palu_2018          105        2018-09-29      2018-10-03              244
sumatra_2025          142        2025-11-29      2025-12-04              366

TEMPORAL PART2 — FINAL AUDIT
Canonical tweet rows       : 348
Unique tweet IDs           : 348
Matching raw occurrences   : 831
Canonical timestamp missing: 0
Timestamp conflicts        : 0
Case distribution preserved: True
Label distribution preserved: True
Raw sources modified       : False
Training performed         : False

ARTIFACTS:
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\timestamp_canonicalization\temporal_labeled_348_locked.csv
- C:\Penelitian Sentimen Bencana Na

In [3]:
# TEMPORAL PART3 — CELL 03
# DAILY SENTIMENT AGGREGATION & INTEGRITY CHECKS
#
# READ-ONLY INPUT | NO MODEL TRAINING | NO SOURCE MODIFICATION
# FULL DATE CALENDAR | CONSERVATION CHECKS | IMMUTABLE EXPORT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# A. PATHS & LOCKED SOURCE CONFIGURATION
# ------------------------------------------------------------

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
PART2 = ROOT / "05_Temporal" / "outputs" / "timestamp_canonicalization"
OUT = ROOT / "05_Temporal" / "outputs" / "daily_aggregation"

DATA_PATH = PART2 / "temporal_labeled_348_locked.csv"
MANIFEST_PATH = PART2 / "temporal_timestamp_lock_manifest.json"
PART2_AUDIT_PATH = PART2 / "part2_timestamp_lock_audit.json"

DAILY_PATH = OUT / "daily_sentiment_counts_20_days.csv"
LONG_PATH = OUT / "daily_sentiment_long_60_rows.csv"
COVERAGE_PATH = OUT / "daily_temporal_coverage.csv"
SUMMARY_PATH = OUT / "part3_daily_aggregation_summary.json"
AUDIT_PATH = OUT / "part3_daily_aggregation_audit.json"

OUTPUTS = [
    DAILY_PATH,
    LONG_PATH,
    COVERAGE_PATH,
    SUMMARY_PATH,
    AUDIT_PATH
]

EXPECTED_DATA_SHA = (
    "4ade3c2d0832f929e0a70ec0ae921653abdcb2aecf715a76fe0c274e674b5267"
)

EXPECTED_CASE_COUNTS = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142
}

EXPECTED_LABEL_COUNTS = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

LABELS = ["NETRAL", "POSITIF", "NEGATIF"]
CASES = ["lombok_2018", "palu_2018", "sumatra_2025"]
TZ = "Asia/Jakarta"

EXPECTED_WINDOWS = {
    "lombok_2018": ("2018-08-06", "2018-08-14"),
    "palu_2018": ("2018-09-29", "2018-10-03"),
    "sumatra_2025": ("2025-11-29", "2025-12-04")
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path, payload):
    path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

print("=" * 76)
print("TEMPORAL PART3 — DAILY SENTIMENT AGGREGATION")
print("=" * 76)

# ------------------------------------------------------------
# B. SOURCE PREFLIGHT & SHA VERIFICATION
# ------------------------------------------------------------

for path in [DATA_PATH, MANIFEST_PATH, PART2_AUDIT_PATH]:
    assert path.is_file(), f"Missing locked source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing Part3 output; refusing overwrite: {path}"
        )

source_sha_before = {
    p.name: sha256_file(p)
    for p in [DATA_PATH, MANIFEST_PATH, PART2_AUDIT_PATH]
}

assert source_sha_before[DATA_PATH.name] == EXPECTED_DATA_SHA

manifest = load_json(MANIFEST_PATH)
audit2 = load_json(PART2_AUDIT_PATH)

assert manifest["status"] == "TEMPORAL_TIMESTAMP_LOCK_FINAL"
assert audit2["status"] == "TEMPORAL_PART2_TIMESTAMP_LOCK_PASS"

assert manifest["output_sha256"][DATA_PATH.name] == EXPECTED_DATA_SHA
assert audit2["output_sha256"][DATA_PATH.name] == EXPECTED_DATA_SHA

assert manifest["canonical_timezone"] == "UTC"
assert manifest["analysis_timezone"] == TZ
assert manifest["canonical_rows"] == 348

print("PART2 LOCK AND SHA256: PASS")

# ------------------------------------------------------------
# C. LOAD & VALIDATE LOCKED TEMPORAL DATASET
# ------------------------------------------------------------

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    dtype={
        "id": "string",
        "case": "string",
        "sentiment_label": "string",
        "leakage_group_id": "string",
        "date_wib": "string",
        "created_at_utc": "string",
        "created_at_wib": "string"
    },
    keep_default_na=False
)

required = {
    "id", "case", "sentiment_label",
    "leakage_group_id", "created_at_utc",
    "created_at_wib", "date_wib"
}

assert required.issubset(df.columns), (
    f"Missing columns: {sorted(required - set(df.columns))}"
)

assert len(df) == 348
assert df["id"].is_unique
assert df["id"].notna().all()
assert df["sentiment_label"].isin(LABELS).all()
assert df["case"].isin(CASES).all()

assert df["case"].value_counts().to_dict() == EXPECTED_CASE_COUNTS
assert (
    df["sentiment_label"].value_counts().to_dict()
    == EXPECTED_LABEL_COUNTS
)

# Validate UTC/WIB conversion independently from date_wib.
utc_dt = pd.to_datetime(
    df["created_at_utc"],
    format="%Y-%m-%dT%H:%M:%S%z",
    utc=True,
    errors="coerce"
)

wib_dt = pd.to_datetime(
    df["created_at_wib"],
    format="%Y-%m-%dT%H:%M:%S%z",
    utc=True,
    errors="coerce"
)

assert utc_dt.notna().all()
assert wib_dt.notna().all()

# Both timestamps must represent exactly the same instant.
assert np.array_equal(
    utc_dt.astype("int64").to_numpy(),
    wib_dt.astype("int64").to_numpy()
)

derived_dates = (
    utc_dt.dt.tz_convert(TZ)
    .dt.strftime("%Y-%m-%d")
)

assert np.array_equal(
    derived_dates.to_numpy(dtype=str),
    df["date_wib"].to_numpy(dtype=str)
), "WIB date does not match canonical UTC timestamp"

assert df["date_wib"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()

# Check dates remain within locked observed case windows.
for case, (start, end) in EXPECTED_WINDOWS.items():
    section = df[df["case"] == case]

    assert section["date_wib"].min() == start
    assert section["date_wib"].max() == end

    assert section["date_wib"].between(start, end).all()

print("348 IDs, LABELS, CASES AND WIB DATES: PASS")

# ------------------------------------------------------------
# D. GENERATE COMPLETE PER-CASE DAILY CALENDAR
# ------------------------------------------------------------

calendar_parts = []

for case in CASES:
    start, end = EXPECTED_WINDOWS[case]

    dates = pd.date_range(
        start=start,
        end=end,
        freq="D"
    )

    calendar_parts.append(pd.DataFrame({
        "case": case,
        "date_wib": dates.strftime("%Y-%m-%d"),
        "relative_day": np.arange(len(dates), dtype=int)
    }))

calendar = pd.concat(
    calendar_parts,
    ignore_index=True
)

assert len(calendar) == 20
assert not calendar.duplicated(
    ["case", "date_wib"]
).any()

# ------------------------------------------------------------
# E. COUNT DAILY SENTIMENTS
# ------------------------------------------------------------

counts = (
    df.groupby(
        ["case", "date_wib", "sentiment_label"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
)

counts = counts.reindex(
    columns=LABELS,
    fill_value=0
)

counts = counts.rename(columns={
    "NETRAL": "count_NETRAL",
    "POSITIF": "count_POSITIF",
    "NEGATIF": "count_NEGATIF"
}).reset_index()

daily = calendar.merge(
    counts,
    on=["case", "date_wib"],
    how="left",
    validate="one_to_one"
)

count_cols = [
    "count_NETRAL",
    "count_POSITIF",
    "count_NEGATIF"
]

daily[count_cols] = (
    daily[count_cols]
    .fillna(0)
    .astype(int)
)

daily["total_tweets"] = daily[count_cols].sum(axis=1)
daily["has_observations"] = daily["total_tweets"] > 0

# Zero-tweet days must remain in the calendar.
daily["is_zero_tweet_day"] = daily["total_tweets"] == 0

# Percentages are undefined on zero-tweet days.
for label in LABELS:
    daily[f"pct_{label}"] = np.where(
        daily["total_tweets"] > 0,
        (
            daily[f"count_{label}"]
            / daily["total_tweets"]
            * 100.0
        ),
        np.nan
    )

daily["dominant_sentiment"] = np.where(
    daily["total_tweets"] == 0,
    "NO_OBSERVATIONS",
    daily[count_cols].idxmax(axis=1)
        .str.replace("count_", "", regex=False)
)

# A tied maximum must not silently become NETRAL.
def dominant_label(row):
    if row["total_tweets"] == 0:
        return "NO_OBSERVATIONS"

    maximum = max(int(row[f"count_{x}"]) for x in LABELS)

    winners = [
        label for label in LABELS
        if int(row[f"count_{label}"]) == maximum
    ]

    return winners[0] if len(winners) == 1 else "TIE"

daily["dominant_sentiment"] = daily.apply(
    dominant_label, axis=1
)

daily = daily[[
    "case", "date_wib", "relative_day",
    "count_NETRAL", "count_POSITIF", "count_NEGATIF",
    "total_tweets",
    "pct_NETRAL", "pct_POSITIF", "pct_NEGATIF",
    "dominant_sentiment",
    "has_observations", "is_zero_tweet_day"
]]

# ------------------------------------------------------------
# F. LONG FORMAT FOR PART4 VISUALIZATION
# ------------------------------------------------------------

long_frames = []

for label in LABELS:
    tmp = daily[[
        "case", "date_wib", "relative_day",
        "total_tweets", f"count_{label}",
        f"pct_{label}", "has_observations"
    ]].copy()

    tmp = tmp.rename(columns={
        f"count_{label}": "sentiment_count",
        f"pct_{label}": "sentiment_pct"
    })

    tmp["sentiment_label"] = label

    long_frames.append(tmp)

daily_long = pd.concat(
    long_frames, ignore_index=True
).sort_values(
    ["case", "date_wib", "sentiment_label"],
    kind="mergesort"
).reset_index(drop=True)

daily_long = daily_long[[
    "case", "date_wib", "relative_day",
    "sentiment_label", "sentiment_count",
    "total_tweets", "sentiment_pct",
    "has_observations"
]]

assert len(daily_long) == 60

# ------------------------------------------------------------
# G. COVERAGE SUMMARY
# ------------------------------------------------------------

coverage_rows = []

for case in CASES:
    case_daily = daily[daily["case"] == case].copy()
    case_raw = df[df["case"] == case]

    coverage_rows.append({
        "case": case,
        "start_date_wib": case_daily["date_wib"].min(),
        "end_date_wib": case_daily["date_wib"].max(),
        "calendar_days": int(len(case_daily)),
        "observed_days": int(
            case_daily["has_observations"].sum()
        ),
        "zero_tweet_days": int(
            case_daily["is_zero_tweet_day"].sum()
        ),
        "total_tweets": int(
            case_daily["total_tweets"].sum()
        ),
        "unique_tweet_ids": int(case_raw["id"].nunique()),
        "count_NETRAL": int(
            case_daily["count_NETRAL"].sum()
        ),
        "count_POSITIF": int(
            case_daily["count_POSITIF"].sum()
        ),
        "count_NEGATIF": int(
            case_daily["count_NEGATIF"].sum()
        )
    })

coverage = pd.DataFrame(coverage_rows)

# ------------------------------------------------------------
# H. STRICT AGGREGATION INTEGRITY CHECKS
# ------------------------------------------------------------

assert len(daily) == 20
assert len(daily_long) == 60

assert int(daily["total_tweets"].sum()) == 348
assert int(daily_long["sentiment_count"].sum()) == 348

assert not daily.duplicated(["case", "date_wib"]).any()

assert not daily_long.duplicated(
    ["case", "date_wib", "sentiment_label"]
).any()

for case in CASES:
    source_case = df[df["case"] == case]
    aggregated_case = daily[daily["case"] == case]

    assert int(aggregated_case["total_tweets"].sum()) == (
        EXPECTED_CASE_COUNTS[case]
    )

    for label in LABELS:
        original_count = int(
            (source_case["sentiment_label"] == label).sum()
        )
        aggregated_count = int(
            aggregated_case[f"count_{label}"].sum()
        )
        assert original_count == aggregated_count, (
            f"Count conservation failed: {case}/{label}"
        )

for label in LABELS:
    assert int(daily[f"count_{label}"].sum()) == (
        EXPECTED_LABEL_COUNTS[label]
    )

observed_days = daily["has_observations"]

percent_total = daily.loc[
    observed_days,
    ["pct_NETRAL", "pct_POSITIF", "pct_NEGATIF"]
].sum(axis=1)

assert np.allclose(
    percent_total.to_numpy(dtype=float),
    100.0,
    atol=1e-9
)

assert daily.loc[
    ~observed_days,
    ["pct_NETRAL", "pct_POSITIF", "pct_NEGATIF"]
].isna().all().all()

assert (daily[count_cols] >= 0).all().all()
assert (daily["total_tweets"] >= 0).all()

assert daily_long.loc[
    ~daily_long["has_observations"],
    "sentiment_pct"
].isna().all()

assert np.array_equal(
    coverage["total_tweets"].to_numpy(),
    coverage["unique_tweet_ids"].to_numpy()
)

assert int(coverage["calendar_days"].sum()) == 20
assert int(coverage["total_tweets"].sum()) == 348

print("DAILY COUNT CONSERVATION: PASS")
print("PER-CASE / PER-LABEL CONSERVATION: PASS")
print("ZERO-TWEET DAY HANDLING: PASS")
print("DAILY PERCENTAGE CHECK: PASS")

# ------------------------------------------------------------
# I. SUMMARY & AUDIT
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

summary = {
    "stage": "05_Temporal",
    "part": "3 Daily Sentiment Aggregation",
    "generated_at_utc": now,
    "source_commit":
        "b48d7164dcf59f6f6d13d1488e593228409669cf",
    "dataset_id": "TEMPORAL_LABELED_348_V1",
    "timezone": TZ,
    "aggregation_unit": "calendar_day",
    "label_source": "locked_human_annotation",
    "calendar_scope": "observed_min_to_max_per_case",
    "source_rows": 348,
    "daily_calendar_rows": int(len(daily)),
    "daily_long_rows": int(len(daily_long)),
    "total_aggregated_tweets": int(
        daily["total_tweets"].sum()
    ),
    "zero_tweet_day_policy":
        "count=0; sentiment_pct=null; no observations",
    "dominant_sentiment_tie_policy": "TIE",
    "source_case_counts": EXPECTED_CASE_COUNTS,
    "source_label_counts": EXPECTED_LABEL_COUNTS,
    "coverage_by_case": coverage.to_dict("records"),
    "model_training_performed": False,
    "status": "TEMPORAL_PART3_AGGREGATION_COMPLETE"
}

audit = {
    "stage": "05_Temporal",
    "part": "3 Daily Aggregation Integrity Audit",
    "generated_at_utc": now,
    "source_commit":
        "b48d7164dcf59f6f6d13d1488e593228409669cf",
    "source_sha256": source_sha_before,
    "source_rows": 348,
    "daily_rows": 20,
    "long_rows": 60,
    "aggregated_count": int(
        daily["total_tweets"].sum()
    ),
    "case_count_conservation": True,
    "class_count_conservation": True,
    "date_wib_reverified_from_utc": True,
    "zero_tweet_dates_preserved": True,
    "undefined_percentages_are_null": True,
    "source_modified": False,
    "training_performed": False,
    "output_sha256": {},
    "status": "TEMPORAL_PART3_DAILY_AGGREGATION_PASS"
}

# ------------------------------------------------------------
# J. IMMUTABLE EXPORT
# ------------------------------------------------------------

# Ensure source is unchanged BEFORE writing.
assert {
    p.name: sha256_file(p)
    for p in [DATA_PATH, MANIFEST_PATH, PART2_AUDIT_PATH]
} == source_sha_before

OUT.mkdir(parents=True, exist_ok=True)

for output in OUTPUTS:
    if output.exists():
        raise FileExistsError(
            f"Refusing overwrite: {output}"
        )

token = uuid.uuid4().hex[:10]
temps = {
    path: OUT / f".part3_{token}_{i}.tmp"
    for i, path in enumerate(OUTPUTS)
}
created = []

try:
    daily.to_csv(
        temps[DAILY_PATH],
        index=False, encoding="utf-8-sig",
        na_rep=""
    )
    daily_long.to_csv(
        temps[LONG_PATH],
        index=False, encoding="utf-8-sig",
        na_rep=""
    )
    coverage.to_csv(
        temps[COVERAGE_PATH],
        index=False, encoding="utf-8-sig"
    )
    save_json(temps[SUMMARY_PATH], summary)

    audit["output_sha256"] = {
        path.name: sha256_file(temps[path])
        for path in [
            DAILY_PATH, LONG_PATH,
            COVERAGE_PATH, SUMMARY_PATH
        ]
    }

    save_json(temps[AUDIT_PATH], audit)

    # Verify persisted CSV counts before publication.
    check_daily = pd.read_csv(
        temps[DAILY_PATH],
        encoding="utf-8-sig"
    )
    check_long = pd.read_csv(
        temps[LONG_PATH],
        encoding="utf-8-sig"
    )

    assert len(check_daily) == 20
    assert len(check_long) == 60
    assert check_daily["total_tweets"].sum() == 348
    assert check_long["sentiment_count"].sum() == 348

    # Published sources must still match locked hashes.
    assert {
        p.name: sha256_file(p)
        for p in [DATA_PATH, MANIFEST_PATH, PART2_AUDIT_PATH]
    } == source_sha_before

    for output in OUTPUTS:
        with output.open("xb") as dest:
            dest.write(temps[output].read_bytes())
        created.append(output)

except Exception:
    for output in created:
        if output.exists():
            output.unlink()
    raise
finally:
    for temp in temps.values():
        if temp.exists():
            temp.unlink()

# ------------------------------------------------------------
# K. FINAL AUDIT & OUTPUT
# ------------------------------------------------------------

for filename, expected_sha in audit["output_sha256"].items():
    assert sha256_file(OUT / filename) == expected_sha

assert {
    p.name: sha256_file(p)
    for p in [DATA_PATH, MANIFEST_PATH, PART2_AUDIT_PATH]
} == source_sha_before

print("\n" + "=" * 76)
print("TEMPORAL PART3 — DAILY COVERAGE")
print("=" * 76)
display(coverage)

print("\nTEMPORAL PART3 — DAILY COUNTS")
display(daily)

print("\nOVERALL SENTIMENT TOTALS")
for label in LABELS:
    print(f"{label:8}: {daily[f'count_{label}'].sum()}")

print("\nTOTAL CALENDAR DAYS :", len(daily))
print("TOTAL LONG ROWS     :", len(daily_long))
print("TOTAL TWEETS        :", daily["total_tweets"].sum())
print("ZERO-TWEET DAYS     :", daily["is_zero_tweet_day"].sum())

print("\nOUTPUT FILES")
for output in OUTPUTS:
    print("-", output)

print("\nFINAL STATUS:", audit["status"])
print("=" * 76)


TEMPORAL PART3 — DAILY SENTIMENT AGGREGATION
PART2 LOCK AND SHA256: PASS
348 IDs, LABELS, CASES AND WIB DATES: PASS
DAILY COUNT CONSERVATION: PASS
PER-CASE / PER-LABEL CONSERVATION: PASS
ZERO-TWEET DAY HANDLING: PASS
DAILY PERCENTAGE CHECK: PASS

TEMPORAL PART3 — DAILY COVERAGE


,case,start_date_wib,end_date_wib,calendar_days,observed_days,zero_tweet_days,total_tweets,unique_tweet_ids,count_NETRAL,count_POSITIF,count_NEGATIF
0,lombok_2018,2018-08-06,2018-08-14,9,6,3,101,101,65,33,3
1,palu_2018,2018-09-29,2018-10-03,5,5,0,105,105,71,29,5
2,sumatra_2025,2025-11-29,2025-12-04,6,4,2,142,142,89,46,7



TEMPORAL PART3 — DAILY COUNTS


,case,date_wib,relative_day,count_NETRAL,count_POSITIF,count_NEGATIF,total_tweets,pct_NETRAL,pct_POSITIF,pct_NEGATIF,dominant_sentiment,has_observations,is_zero_tweet_day
0,lombok_2018,2018-08-06,0,5,5,0,10,50.000000,50.000000,0.000000,TIE,True,False
1,lombok_2018,2018-08-07,1,0,0,0,0,NaN,NaN,NaN,NO_OBSERVATIONS,False,True
2,lombok_2018,2018-08-08,2,0,0,0,0,NaN,NaN,NaN,NO_OBSERVATIONS,False,True
3,lombok_2018,2018-08-09,3,0,0,0,0,NaN,NaN,NaN,NO_OBSERVATIONS,False,True
4,lombok_2018,2018-08-10,4,13,11,0,24,54.166667,45.833333,0.000000,NETRAL,True,False
5,lombok_2018,2018-08-11,5,17,3,1,21,80.952381,14.285714,4.761905,NETRAL,True,False
6,lombok_2018,2018-08-12,6,12,4,0,16,75.000000,25.000000,0.000000,NETRAL,True,False
7,lombok_2018,2018-08-13,7,12,7,1,20,60.000000,35.000000,5.000000,NETRAL,True,False
8,lombok_2018,2018-08-14,8,6,3,1,10,60.000000,30.000000,10.000000,NETRAL,True,False
9,palu_2018,2018-09-29,0,2,7,0,9,22.222222,77.777778,0.000000,POSITIF,True,False



OVERALL SENTIMENT TOTALS
NETRAL  : 225
POSITIF : 108
NEGATIF : 15

TOTAL CALENDAR DAYS : 20
TOTAL LONG ROWS     : 60
TOTAL TWEETS        : 348
ZERO-TWEET DAYS     : 5

OUTPUT FILES
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\daily_aggregation\daily_sentiment_counts_20_days.csv
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\daily_aggregation\daily_sentiment_long_60_rows.csv
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\daily_aggregation\daily_temporal_coverage.csv
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\daily_aggregation\part3_daily_aggregation_summary.json
- C:\Penelitian Sentimen Bencana Nasional\05_Temporal\outputs\daily_aggregation\part3_daily_aggregation_audit.json

FINAL STATUS: TEMPORAL_PART3_DAILY_AGGREGATION_PASS


In [4]:
# TEMPORAL PART4 — CELL 04
# SENTIMENT TIMELINE VISUALIZATION & FIGURE AUDIT
#
# READ-ONLY SOURCES | NO TRAINING | NO OVERWRITE
# 8 PNG FIGURES + REGISTRY + SUMMARY + SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
BASE = ROOT / "05_Temporal" / "outputs"
SOURCE_DIR = BASE / "daily_aggregation"
OUT = BASE / "timeline_visualizations"

DAILY_PATH = SOURCE_DIR / "daily_sentiment_counts_20_days.csv"
LONG_PATH = SOURCE_DIR / "daily_sentiment_long_60_rows.csv"
COVERAGE_PATH = SOURCE_DIR / "daily_temporal_coverage.csv"
SUMMARY3_PATH = SOURCE_DIR / "part3_daily_aggregation_summary.json"
AUDIT3_PATH = SOURCE_DIR / "part3_daily_aggregation_audit.json"

EXPECTED_SHA = {
    DAILY_PATH.name:
        "4e4e1fd9e8b4e25b1aef729905b81306dd55a0d8053a2b509cee5d8acf89df21",
    LONG_PATH.name:
        "2458f2a2fbfc3dcf2c8481233fa480685f130153d3dddd721b99fb5aa46e54c9",
    COVERAGE_PATH.name:
        "aae38c63e7c155067f5773f370efdb315566a8ac0d4450999b8f74eeefa423f1",
    SUMMARY3_PATH.name:
        "53ce375c9e22416cc3cd62495164d533d298070b6806aae77a6ce66adf92377d",
}

CASES = [
    "lombok_2018",
    "palu_2018",
    "sumatra_2025",
]

CASE_NAMES = {
    "lombok_2018": "Lombok 2018",
    "palu_2018": "Palu 2018",
    "sumatra_2025": "Sumatera 2025",
}

LABELS = ["NETRAL", "POSITIF", "NEGATIF"]

COLORS = {
    "NETRAL": "#64748b",
    "POSITIF": "#16a34a",
    "NEGATIF": "#dc2626",
}

FIGURES = [
    ("01_lombok_daily_counts.png", "lombok_2018", "daily_counts"),
    ("02_palu_daily_counts.png", "palu_2018", "daily_counts"),
    ("03_sumatra_daily_counts.png", "sumatra_2025", "daily_counts"),
    ("04_lombok_daily_percentage.png", "lombok_2018", "daily_percentage"),
    ("05_palu_daily_percentage.png", "palu_2018", "daily_percentage"),
    ("06_sumatra_daily_percentage.png", "sumatra_2025", "daily_percentage"),
    ("07_relative_day_tweet_volume.png", "all_cases", "relative_volume"),
    ("08_sentiment_composition_by_case.png", "all_cases", "case_composition"),
]

REGISTRY_PATH = OUT / "temporal_figure_registry.csv"
SUMMARY_PATH = OUT / "part4_visualization_summary.json"
AUDIT_PATH = OUT / "part4_figure_audit.json"

OUTPUTS = (
    [OUT / name for name, _, _ in FIGURES]
    + [REGISTRY_PATH, SUMMARY_PATH, AUDIT_PATH]
)

SOURCES = [
    DAILY_PATH, LONG_PATH, COVERAGE_PATH,
    SUMMARY3_PATH, AUDIT3_PATH
]

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, obj):
    path.write_text(
        json.dumps(
            obj, ensure_ascii=False,
            indent=2, allow_nan=False
        ),
        encoding="utf-8"
    )

print("=" * 75)
print("TEMPORAL PART4 — FIGURE PREFLIGHT")
print("=" * 75)

# ------------------------------------------------------------
# A. VERIFY IMMUTABLE PART3 SOURCES
# ------------------------------------------------------------

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Output exists; refusing overwrite: {path}"
        )

source_before = {
    p.name: sha256_file(p)
    for p in SOURCES
}

for name, expected in EXPECTED_SHA.items():
    assert source_before[name] == expected, (
        f"Locked source SHA mismatch: {name}"
    )

summary3 = read_json(SUMMARY3_PATH)
audit3 = read_json(AUDIT3_PATH)

assert summary3["status"] == (
    "TEMPORAL_PART3_AGGREGATION_COMPLETE"
)
assert audit3["status"] == (
    "TEMPORAL_PART3_DAILY_AGGREGATION_PASS"
)

for name in [
    DAILY_PATH.name, LONG_PATH.name,
    COVERAGE_PATH.name, SUMMARY3_PATH.name
]:
    assert audit3["output_sha256"][name] == source_before[name]

daily = pd.read_csv(DAILY_PATH, encoding="utf-8-sig")
long_df = pd.read_csv(LONG_PATH, encoding="utf-8-sig")
coverage = pd.read_csv(COVERAGE_PATH, encoding="utf-8-sig")

assert len(daily) == 20
assert len(long_df) == 60
assert len(coverage) == 3
assert daily["total_tweets"].sum() == 348
assert long_df["sentiment_count"].sum() == 348
assert not daily.duplicated(["case", "date_wib"]).any()
assert not long_df.duplicated(
    ["case", "date_wib", "sentiment_label"]
).any()

assert set(daily["case"]) == set(CASES)
assert set(long_df["sentiment_label"]) == set(LABELS)

assert int(daily["is_zero_tweet_day"].sum()) == 5

# Daily percentage columns: NaN is mandatory
# when total_tweets == 0.
percentage_cols = [f"pct_{x}" for x in LABELS]

observed = daily["total_tweets"] > 0

assert daily.loc[
    ~observed, percentage_cols
].isna().all().all()

assert np.allclose(
    daily.loc[observed, percentage_cols]
    .sum(axis=1).to_numpy(dtype=float),
    100.0, atol=1e-9
)

# Independently cross-check long format.
for label in LABELS:
    wide_label = (
        daily[["case", "date_wib", f"count_{label}"]]
        .rename(columns={f"count_{label}": "wide_count"})
    )
    long_label = long_df[
        long_df["sentiment_label"] == label
    ][["case", "date_wib", "sentiment_count"]]

    check = wide_label.merge(
        long_label,
        on=["case", "date_wib"],
        validate="one_to_one"
    )
    assert len(check) == 20
    assert (
        check["wide_count"] == check["sentiment_count"]
    ).all()

print("PART3 SHA256: PASS")
print("DAILY/LONG RECONCILIATION: PASS")
print("ZERO-TWEET PERCENTAGE HANDLING: PASS")

# ------------------------------------------------------------
# B. GRAPH STYLE & HELPER
# ------------------------------------------------------------

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def prepare_case(case):
    frame = (
        daily[daily["case"] == case]
        .sort_values("relative_day")
        .copy()
    )
    assert frame["relative_day"].tolist() == list(
        range(len(frame))
    )
    return frame

def finish_figure(fig, path):
    fig.savefig(
        path, dpi=220,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

def plot_daily_counts(case, destination):
    frame = prepare_case(case)
    x = np.arange(len(frame))
    bottom = np.zeros(len(frame), dtype=int)

    fig, ax = plt.subplots(figsize=(11, 5.5))

    for label in LABELS:
        values = frame[f"count_{label}"].to_numpy()
        ax.bar(
            x, values, bottom=bottom,
            color=COLORS[label], label=label,
            width=0.7
        )
        bottom += values

    for i, count in enumerate(frame["total_tweets"]):
        if count == 0:
            ax.text(
                i, 0.3, "Tanpa\nobservasi",
                ha="center", va="bottom",
                fontsize=8, color="#555555"
            )
        else:
            ax.text(
                i, count + 0.5, f"n={count}",
                ha="center", fontsize=8
            )

    ax.set_xticks(x)
    ax.set_xticklabels(
        frame["date_wib"].str.slice(5),
        rotation=0
    )
    ax.set_xlabel("Tanggal publikasi tweet (WIB, MM-DD)")
    ax.set_ylabel("Jumlah tweet")
    ax.set_title(
        f"Jumlah Sentimen Harian — {CASE_NAMES[case]}"
    )
    ax.set_ylim(0, max(frame["total_tweets"]) * 1.23 + 2)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(ncol=3, loc="upper right")
    fig.tight_layout()
    finish_figure(fig, destination)

def plot_daily_percentages(case, destination):
    frame = prepare_case(case)
    x = frame["relative_day"].to_numpy()

    fig, ax = plt.subplots(figsize=(11, 5.5))

    for label in LABELS:
        y = frame[f"pct_{label}"].to_numpy(
            dtype=float
        )

        # NaN causes matplotlib to BREAK the line.
        # Do not interpolate across missing dates.
        ax.plot(
            x, y,
            color=COLORS[label],
            marker="o",
            linewidth=2,
            label=label
        )

    for i, row in enumerate(frame.itertuples()):
        if row.total_tweets == 0:
            ax.axvspan(
                i - 0.4, i + 0.4,
                color="#dddddd", alpha=0.35
            )
            ax.text(
                i, 6, "n=0",
                ha="center", fontsize=8
            )
        else:
            ax.text(
                i, 104, f"n={row.total_tweets}",
                ha="center", fontsize=8
            )

    ax.set_xticks(x)
    ax.set_xticklabels(frame["date_wib"].str.slice(5))
    ax.set_ylim(0, 112)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_ylabel("Proporsi sentimen (%)")
    ax.set_xlabel("Tanggal publikasi tweet (WIB, MM-DD)")
    ax.set_title(
        f"Proporsi Sentimen Harian — {CASE_NAMES[case]}"
    )
    ax.grid(axis="y", alpha=0.2)
    ax.legend(ncol=3, loc="upper right")
    fig.text(
        0.5, 0.01,
        "Hari tanpa tweet: proporsi tidak terdefinisi; garis tidak diinterpolasi.",
        ha="center", fontsize=8, color="#555555"
    )
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    finish_figure(fig, destination)

def plot_relative_volume(destination):
    fig, ax = plt.subplots(figsize=(11, 5.5))

    case_colors = {
        "lombok_2018": "#2563eb",
        "palu_2018": "#d97706",
        "sumatra_2025": "#7c3aed",
    }

    for case in CASES:
        frame = prepare_case(case)

        ax.plot(
            frame["relative_day"],
            frame["total_tweets"],
            marker="o", linewidth=2,
            color=case_colors[case],
            label=CASE_NAMES[case]
        )

    ax.set_xlabel(
        "Hari relatif sejak awal jendela observasi"
    )
    ax.set_ylabel("Jumlah tweet")
    ax.set_title(
        "Volume Tweet Harian Menurut Hari Relatif Observasi"
    )
    ax.set_xticks(range(9))
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.text(
        0.5, 0.01,
        "Hari 0 = tanggal pertama observasi, BUKAN tanggal kejadian bencana.",
        ha="center", fontsize=8, color="#555555"
    )
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    finish_figure(fig, destination)

def plot_case_composition(destination):
    grouped = (
        daily.groupby("case")[
            [f"count_{label}" for label in LABELS]
        ].sum()
        .reindex(CASES)
    )
    totals = grouped.sum(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    x = np.arange(len(CASES))
    bottom = np.zeros(len(CASES), dtype=float)

    for label in LABELS:
        pct = (
            grouped[f"count_{label}"]
            / totals * 100
        ).to_numpy(dtype=float)

        ax.bar(
            x, pct, bottom=bottom,
            color=COLORS[label], label=label,
            width=0.6
        )

        for i, value in enumerate(pct):
            if value >= 4:
                ax.text(
                    i, bottom[i] + value / 2,
                    f"{value:.1f}%",
                    ha="center", va="center",
                    fontsize=9, color="white"
                )
        bottom += pct

    for i, count in enumerate(totals):
        ax.text(
            i, 102, f"n={count}",
            ha="center", fontsize=9
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [CASE_NAMES[case] for case in CASES]
    )
    ax.set_ylim(0, 110)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_ylabel("Proporsi sampel tweet (%)")
    ax.set_title(
        "Komposisi Sentimen Seluruh Sampel per Kasus"
    )
    ax.legend(ncol=3, loc="upper right")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    fig.tight_layout()
    finish_figure(fig, destination)

# ------------------------------------------------------------
# C. GENERATE FIGURES INTO TEMP FILES
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

token = uuid.uuid4().hex[:10]

temp_paths = {
    path: OUT / f".part4_{token}_{i}.png"
    if path.suffix == ".png"
    else OUT / f".part4_{token}_{i}.tmp"
    for i, path in enumerate(OUTPUTS)
}

created = []

try:
    registry_rows = []

    for number, (name, case, chart_type) in enumerate(
        FIGURES, start=1
    ):
        final_path = OUT / name
        temporary = temp_paths[final_path]

        if chart_type == "daily_counts":
            plot_daily_counts(case, temporary)
        elif chart_type == "daily_percentage":
            plot_daily_percentages(case, temporary)
        elif chart_type == "relative_volume":
            plot_relative_volume(temporary)
        elif chart_type == "case_composition":
            plot_case_composition(temporary)
        else:
            raise ValueError(
                f"Unknown chart type: {chart_type}"
            )

        assert temporary.is_file()
        assert temporary.stat().st_size > 1000
        assert temporary.read_bytes()[:8] == (
            bytes.fromhex("89504e470d0a1a0a")
        )

        registry_rows.append({
            "figure_id": f"T04_F{number:02d}",
            "filename": name,
            "case": case,
            "chart_type": chart_type,
            "aggregation_unit": "calendar_day_wib",
            "source_dataset": DAILY_PATH.name,
            "source_sha256": source_before[DAILY_PATH.name],
            "figure_sha256": sha256_file(temporary),
            "file_size_bytes": temporary.stat().st_size,
            "includes_daily_sample_size": (
                chart_type in [
                    "daily_counts", "daily_percentage",
                    "case_composition"
                ]
            ),
            "missing_day_policy": (
                "NaN gap; no interpolation"
                if chart_type == "daily_percentage"
                else "count zero retained"
            )
        })

    registry = pd.DataFrame(registry_rows)

    assert len(registry) == 8
    assert registry["figure_sha256"].nunique() == 8

    registry.to_csv(
        temp_paths[REGISTRY_PATH],
        index=False,
        encoding="utf-8-sig"
    )

    coverage_rows = []

    for case in CASES:
        frame = prepare_case(case)
        coverage_rows.append({
            "case": case,
            "calendar_days": len(frame),
            "observed_days": int(
                (frame["total_tweets"] > 0).sum()
            ),
            "zero_tweet_days": int(
                (frame["total_tweets"] == 0).sum()
            ),
            "total_tweets": int(
                frame["total_tweets"].sum()
            ),
            "peak_observed_date": str(
                frame.loc[
                    frame["total_tweets"].idxmax(),
                    "date_wib"
                ]
            ),
            "peak_daily_tweet_count": int(
                frame["total_tweets"].max()
            )
        })

    result_summary = {
        "stage": "05_Temporal",
        "part": "4 Sentiment Timeline Visualization",
        "generated_at_utc":
            datetime.now(timezone.utc).isoformat(),
        "source_commit":
            "664de8ba4ebca944a4a5f529848f18122f6691d4",
        "source_part3_status":
            "TEMPORAL_PART3_DAILY_AGGREGATION_PASS",
        "figure_count": 8,
        "daily_rows": 20,
        "long_rows": 60,
        "total_tweets": 348,
        "daily_zero_tweet_days": 5,
        "timezone": "Asia/Jakarta",
        "relative_day_definition":
            "Days since first observed date per case",
        "source_labels": "locked_human_annotation",
        "percentage_gap_policy":
            "Undefined on zero-observation dates; no interpolation",
        "case_coverage": coverage_rows,
        "model_training_performed": False,
        "status": "TEMPORAL_PART4_FIGURES_COMPLETE"
    }

    write_json(
        temp_paths[SUMMARY_PATH],
        result_summary
    )

    result_audit = {
        "stage": "05_Temporal",
        "part": "4 Timeline Figure Audit",
        "generated_at_utc":
            datetime.now(timezone.utc).isoformat(),
        "source_commit":
            "664de8ba4ebca944a4a5f529848f18122f6691d4",
        "source_sha256": source_before,
        "figure_count": 8,
        "valid_png_count": 8,
        "zero_observation_days_retained": True,
        "undefined_daily_percentages_not_interpolated": True,
        "source_dataset_modified": False,
        "model_training_performed": False,
        "output_sha256": {},
        "status": "TEMPORAL_PART4_FIGURE_AUDIT_PASS"
    }

    result_audit["output_sha256"] = {
        path.name: sha256_file(temp_paths[path])
        for path in OUTPUTS[:-1]
    }

    write_json(
        temp_paths[AUDIT_PATH],
        result_audit
    )

    # Input provenance must not change.
    assert {
        p.name: sha256_file(p)
        for p in SOURCES
    } == source_before

    # Publish without overwriting existing files.
    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(temp_paths[destination].read_bytes())
        created.append(destination)

except Exception:
    # Delete only files created by THIS execution.
    for destination in created:
        if destination.exists():
            destination.unlink()
    raise

finally:
    for temporary in temp_paths.values():
        if temporary.exists():
            temporary.unlink()
    plt.close("all")

# ------------------------------------------------------------
# D. FINAL AUDIT
# ------------------------------------------------------------

for filename, expected_sha in (
    result_audit["output_sha256"].items()
):
    assert sha256_file(OUT / filename) == expected_sha

assert {
    p.name: sha256_file(p)
    for p in SOURCES
} == source_before

print("\n" + "=" * 75)
print("TEMPORAL PART4 — FINAL AUDIT")
print("=" * 75)
print("Figures generated     :", len(registry))
print("Valid PNG figures     :", 8)
print("Daily calendar rows   :", len(daily))
print("Long-format rows      :", len(long_df))
print("Total tweets          :", int(daily["total_tweets"].sum()))
print("Zero-observation days :", int(
    (daily["total_tweets"] == 0).sum()
))
print("Source data modified  : NO")
print("Model training        : NO")

print("\nCASE COVERAGE:")
display(pd.DataFrame(coverage_rows))

print("\nFIGURE REGISTRY:")
display(registry[[
    "figure_id", "filename",
    "case", "figure_sha256"
]])

print("\nFINAL STATUS:", result_audit["status"])
print("=" * 75)


TEMPORAL PART4 — FIGURE PREFLIGHT
PART3 SHA256: PASS
DAILY/LONG RECONCILIATION: PASS
ZERO-TWEET PERCENTAGE HANDLING: PASS

TEMPORAL PART4 — FINAL AUDIT
Figures generated     : 8
Valid PNG figures     : 8
Daily calendar rows   : 20
Long-format rows      : 60
Total tweets          : 348
Zero-observation days : 5
Source data modified  : NO
Model training        : NO

CASE COVERAGE:


,case,calendar_days,observed_days,zero_tweet_days,total_tweets,peak_observed_date,peak_daily_tweet_count
0,lombok_2018,9,6,3,101,2018-08-10,24
1,palu_2018,5,5,0,105,2018-10-01,36
2,sumatra_2025,6,4,2,142,2025-12-03,54



FIGURE REGISTRY:


,figure_id,filename,case,figure_sha256
0,T04_F01,01_lombok_daily_counts.png,lombok_2018,b57c6e2ab57e506a39eaaceb476325b92a2f699ab39260...
1,T04_F02,02_palu_daily_counts.png,palu_2018,36f8f1a3ea9b7572ea56a85bfffbcf5d547decd2722d60...
2,T04_F03,03_sumatra_daily_counts.png,sumatra_2025,70c7c7377bf05929c5e95d05c3d20b1910ec4aef7d3793...
3,T04_F04,04_lombok_daily_percentage.png,lombok_2018,968233f3d672eac4f93cf46215fb021cc65aa8253215cf...
4,T04_F05,05_palu_daily_percentage.png,palu_2018,5de7458ac2ff99816969687737dd7d21a18581f005d038...
5,T04_F06,06_sumatra_daily_percentage.png,sumatra_2025,cdc95e819b2e79de64ccfecff3206bcaf0e995e959d118...
6,T04_F07,07_relative_day_tweet_volume.png,all_cases,f372e20f3c19becfdb8cef652a1adc44ed48713e7f4e03...
7,T04_F08,08_sentiment_composition_by_case.png,all_cases,2870e85c9791ee31b4c8cbba90f582602c052cc4e55676...



FINAL STATUS: TEMPORAL_PART4_FIGURE_AUDIT_PASS
